In [7]:
import chaospy
import numpy as np

# 1. Store the original numpy.reshape function
_original_numpy_reshape = np.reshape

# 2. Define a wrapper to catch the incorrect 'shape' keyword and replace it with 'newshape'
def patched_numpy_reshape(a, newshape=None, **kwargs):
    if 'shape' in kwargs:
        newshape = kwargs.pop('shape')
    return _original_numpy_reshape(a, newshape, **kwargs)

# 3. Apply the patch globally
np.reshape = patched_numpy_reshape

# ---------------------------------------------------------
# 1. Load Pre-acquired Experimental Data
# ---------------------------------------------------------
n_samples = 5000
n_variables = 2

# Simulating the raw laboratory data (e.g., values between 10 and 50)
X_data_raw = np.random.uniform(10, 50, (n_variables, n_samples))
Y_data = 25 * X_data_raw[0] + 0.1 * X_data_raw[1]**2 # + X_data_raw[2] * np.sin(X_data_raw[0])

# ---------------------------------------------------------
# 2. Scale the Inputs to [-1, 1] (The Fix)
# ---------------------------------------------------------
# Find min and max for each variable (keeping dimensions for broadcasting)
X_min = np.min(X_data_raw, axis=1, keepdims=True)
X_max = np.max(X_data_raw, axis=1, keepdims=True)

Y_min = np.min(Y_data, keepdims=True)
Y_max = np.max(Y_data, keepdims=True)

# Apply Min-Max mapping to standard domain [-1, 1]
X_scaled = 2.0 * (X_data_raw - X_min) / (X_max - X_min) - 1.0
Y_scaled = 2.0 * (Y_data - Y_min) / (Y_max - Y_min) - 1.0

# ---------------------------------------------------------
# 3. Define Standard Distributions
# ---------------------------------------------------------
# Because the data is now scaled, we use standard Uniform(-1, 1) distributions
dist_standard = chaospy.Uniform(-1, 1)
joint_dist = chaospy.Iid(chaospy.Uniform(-1, 1), n_variables)

# ---------------------------------------------------------
# 4. Generate Basis and Fit Model on Scaled Data
# ---------------------------------------------------------
poly_order = 5
expansion = chaospy.generate_expansion(poly_order, joint_dist)

In [8]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1,2)
axs[0].scatter(X_scaled[0], Y_scaled)
axs[1].scatter(X_scaled[1], Y_scaled)

RecursionError: maximum recursion depth exceeded

In [4]:
print(expansion)

print(len(expansion))

[1.0 59.0*q1 59.0*q0 3481.0*q1**2-19.666666666666664 3481.0*q0*q1
 3481.0*q0**2-19.666666666666664 205379.0*q1**3-2088.6*q1
 205379.0*q0*q1**2-1160.3333333333333*q0
 205379.0*q0**2*q1-1160.3333333333333*q1 205379.0*q0**3-2088.6*q0]
10


In [5]:
# Fit using the SCALED inputs
print("Fitting standardized PCE model...")
pce_model = chaospy.fit_regression(expansion, X_scaled, Y_scaled)

# ---------------------------------------------------------
# 5. Extract Analytical Sobol' Indices
# ---------------------------------------------------------
s1 = chaospy.Sens_m(pce_model, joint_dist)
st = chaospy.Sens_t(pce_model, joint_dist)

print("\n--- Sensitivity Results ---")
for i in range(n_variables):
    print(f"Variable {i+1} - First-Order (S1): {s1[i]:.4f} | Total-Order (ST): {st[i]:.4f}")

Fitting standardized PCE model...

--- Sensitivity Results ---
Variable 1 - First-Order (S1): 0.0000 | Total-Order (ST): 0.0000
Variable 2 - First-Order (S1): 0.0000 | Total-Order (ST): 0.0000


In [6]:
pce_model

polynomial(3.5627577277264066e-13*q1**3-3.785430085709307e-13*q0*q1**2+6.680170739487012e-14*q0**2*q1-3.228749190752056e-13*q0**3+0.06595486915591667*q1**2-3.3619044302450087e-14*q0*q1-4.227000693912686e-14*q0**2+0.19793718041170788*q1+0.824424972329221*q0-0.0577240234291383)